# Two-stage coarsening: Star → Edge BPE → exact decode

This tutorial applies two schema-1 stages to a small in-memory tree:

1. `ParametricStarCoarsener` contracts repeated sibling stars.
2. `EdgeBPECoarsener` then learns a repeated edge on the star-coarsened graph.

Decoding reverses the stages in the opposite order: first BPE, then Star. The final
cell also shows the equivalent lazy combined encoder/decoder.

In [ ]:
import networkx as nx

from tree_coarsening import EdgeBPECoarsener, ParametricStarCoarsener, combine

## 1. Build a small raw tree

The tree has three anchor branches. Each anchor contains a `P` node with four `S`
children. Star coarsening will contract each sibling run of `S` children; then Edge
BPE can learn the repeated `P → star(P, S)` edge.

In [ ]:
def make_three_star_tree() -> nx.DiGraph:
    graph = nx.DiGraph()
    graph.add_node(0, label="ROOT", time=0.0, uid="root")
    next_node = 1

    for branch in range(3):
        anchor = next_node
        next_node += 1
        graph.add_node(anchor, label=f"A{branch}", time=1.0 + branch, uid=f"a{branch}")
        graph.add_edge(0, anchor)

        parent = next_node
        next_node += 1
        graph.add_node(parent, label="P", time=1.5 + branch, uid=f"p{branch}")
        graph.add_edge(anchor, parent)

        for child_i in range(4):
            child = next_node
            next_node += 1
            graph.add_node(
                child,
                label="S",
                time=2.0 + branch + 0.1 * child_i,
                uid=f"p{branch}_s{child_i}",
            )
            graph.add_edge(parent, child)

    return graph


def raw_signature(graph: nx.DiGraph):
    nodes = sorted(
        (data["uid"], data["label"], data["time"])
        for _, data in graph.nodes(data=True)
    )
    edges = sorted(
        (graph.nodes[parent]["uid"], graph.nodes[child]["uid"])
        for parent, child in graph.edges
    )
    return nodes, edges, dict(graph.graph)


raw = make_three_star_tree()
print(f"raw tree: {raw.number_of_nodes()} nodes, {raw.number_of_edges()} edges")

## 2. Stage 1: fit and apply the star coarsener

The learned star label is a matching identity. The exact arity and UID provenance are
stored on each occurrence, so decoding can restore the original raw nodes exactly.

In [ ]:
star = ParametricStarCoarsener(
    d=3,
    m=1,
    model_id="star-stage",
).fit([raw])

starred = star.transform(raw)

print(f"after Star: {starred.number_of_nodes()} nodes, {starred.number_of_edges()} edges")
for rule in star.encoder_.rules:
    print(rule.pattern, "->", rule.output_label)

## 3. Stage 2: fit and apply Edge BPE on the Star output

The second stage consumes the first stage's encoded graph directly. It does not need a
special adapter; both stages share the same schema-1 graph contract.

In [ ]:
bpe = EdgeBPECoarsener(
    num_merges=1,
    min_pair_count=2,
    model_id="bpe-stage",
).fit([starred])

final = bpe.transform(starred)

print(f"after Star → BPE: {final.number_of_nodes()} nodes, {final.number_of_edges()} edges")
for row in bpe.history_:
    print(
        f"rank={row['rank']}: {row['parent_label']!r} -> {row['child_label']!r} "
        f"count={row['count']} actual_events={row['actual_events']}"
    )

## 4. Decode in reverse stage order

Decoding is stage-local. Undo the latest stage first, then continue backward through
the pipeline.

In [ ]:
recovered_starred = bpe.decode(final)
recovered_raw = star.decode(recovered_starred)

assert nx.utils.graphs_equal(recovered_starred, starred)
assert raw_signature(recovered_raw) == raw_signature(raw)

print("BPE decode restored the Star graph.")
print("Star decode restored the raw graph.")

## 5. Use a combined encoder/decoder for repeated application

Once individual stages are fitted, `combine(...)` builds a lazy pipeline artifact.
It applies the fitted encoders in order and decodes them in reverse order.

In [ ]:
combined_encoder, combined_decoder = combine(
    [star.encoder_, bpe.encoder_],
    [star.decoder_, bpe.decoder_],
)

combined = combined_encoder.transform(raw)
combined_recovered = combined_decoder.decode(combined)

assert nx.utils.graphs_equal(combined, final)
assert raw_signature(combined_recovered) == raw_signature(raw)

print("combined encoder matches manual Star → BPE application")
print("combined decoder restores the raw graph")